In [8]:
import pandas as pd
import matplotlib.pyplot as plt

In [9]:
# Number of top-ranked movies to keep for each user (change as needed)
M = 10

ratings = pd.read_csv("../data/raw/ratings_small.csv")

In [10]:
# Sort by rating in descending order; for equal ratings,
# put the most recently rated movie first.
selected = ratings.sort_values(
    ["userId", "rating", "timestamp"], ascending=[True, False, False]
).copy()

# Keep only users who rated at least two movies
selected = selected[selected.groupby("userId")["movieId"].transform("size") >= 2].copy()

# Assign the position in the list: 1 = favorite movie
selected["rank"] = selected.groupby("userId").cumcount() + 1

# Keep only the top-M movies of each user
top_m = selected[selected["rank"] <= M]

In [11]:
# Build one row per user: userId + ordered list of their top-M movieIds
preference_lists = (
    top_m.groupby("userId")["movieId"].apply(list).reset_index(name="top_movies")
)

# Save the dataset (the file name includes M, so different values don't overwrite each other)
preference_lists.to_csv(f"../data/processed/preference_lists_top{M}.csv", index=False)

print(preference_lists.head(10))

   userId                                         top_movies
0       1  [1172, 1953, 2105, 1339, 2150, 1061, 1029, 367...
1       2   [222, 585, 551, 265, 589, 17, 39, 266, 150, 590]
2       3  [1197, 3949, 2959, 356, 318, 1721, 50068, 296,...
3       4  [1334, 2454, 596, 1022, 1028, 1033, 2087, 364,...
4       5  [33166, 597, 2081, 1035, 1380, 4025, 4447, 307...
5       6  [293, 7153, 5952, 1204, 1259, 2761, 8874, 1285...
6       7  [260, 1198, 1196, 1210, 1242, 1225, 110, 318, ...
7       8  [2918, 1961, 2329, 32, 6874, 4226, 2502, 47, 2...
8       9  [1059, 36, 1784, 527, 608, 1682, 2571, 534, 23...
9      10  [2926, 1611, 2344, 1923, 2571, 1719, 50, 1358,...


In [12]:
# Show the list of the first available user
first_user = preference_lists.iloc[0]
print(first_user["userId"], first_user["top_movies"])

1 [1172, 1953, 2105, 1339, 2150, 1061, 1029, 3671, 31, 1371]


## Assunzioni necessarie per costruire il dataset preferenziale:

 userId | list of movieId

- Che assunzioni facciamo sui voti dati tanto tempo fa (timestamp)? Le preferenze dell'utente potrebbero essere cambiate nel tempo
- Per l'utente x consideriamo i film con i voti più alti come le prime preferenze dell’utente sull’intero catalogo (l’assenza di un voto non ci dice se il film sarebbe piaciuto più o meno di quelli valutati). Quindi usiamo queste liste come approssimazione delle top-\(m\) preferenze richieste dal modello

Film con stesso voto, idee:
- Teniamo il voto più recente?
- Li teniamo entrambi e mettiamo prima quello recensito più recentemente?
- Li teniamo entrambi e li ordiniamo casualmente?

Quanti m-movies teniamo?